[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/manipulability/manipulability.ipynb)

# Velocity and Force Ellipsoids of an Arm

This example computes the velocity and force manipulability ellipsoids at the gripper of an articulated robot arm, introduced by Yoshikawa (1985). Turning the joints at unit rates moves the gripper tip with velocities that fill an ellipsoid, elongated along directions the arm sweeps easily and flattened along directions it can barely reach. Resisting loads with unit joint torques supports tip forces that fill a reciprocal ellipsoid, elongated wherever the velocity ellipsoid is flat. As the arm stretches nearly straight, the velocity ellipsoid flattens across the arm while the force ellipsoid stretches along it, because an extended arm cannot push its tip farther outward yet carries axial loads through its links with negligible joint torque.

Each joint's rotation generator carried into the world by its cumulative pivot motor `pivots >> joint_generators` is that joint's twist per unit rate. Taking the commutator `axes.commutator(tip)` gives each joint's tip velocity direction, whose dyads sum to the velocity ellipsoid as a dual quadric from planes to points, while joining `axes & tip` gives the plane of tip forces that exert zero torque on each joint, whose dyads sum to the force ellipsoid as a point quadric from points to planes. By virtual work, conjugating through the tip's unit sphere `sphere(velocity(sphere(Point)))` maps the velocity ellipsoid directly onto the force ellipsoid; the articulated arm geometry comes from [scenegraph](../../geometry/scenegraph/scenegraph.ipynb).

In the matrix notation of robotics, each joint axis unit vector $\hat{\mathbf{z}}_i$ and origin $\mathbf{p}_{i-1}$ are shifted to the gripper tip $\mathbf{p}_e$ via cross products $\mathbf{j}_{v,i} = \hat{\mathbf{z}}_i \times (\mathbf{p}_e - \mathbf{p}_{i-1})$ to assemble the $3 \times n$ translational Jacobian $J_v(\mathbf{q})$, giving the velocity ellipsoid $\dot{\mathbf{x}}^{\mathsf T} (J_v J_v^{\mathsf T})^{-1} \dot{\mathbf{x}} \le 1$ from $\dot{\mathbf{x}} = J_v \dot{\mathbf{q}}$ and the force ellipsoid $\mathbf{f}^{\mathsf T} (J_v J_v^{\mathsf T}) \mathbf{f} \le 1$ from the transposed torque map $\boldsymbol{\tau} = J_v^{\mathsf T} \mathbf{f}$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA3D as ga
from examples.util.animation import save_figure, show_states
from examples.mechanics.manipulability import render, scenarios
from examples.geometry.scenegraph import core as arm

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Plane = ga.gatype.vector()                         # Vector+
Point = ga.gatype.antivector()                     # Antivector-
PointMap = ga.gatype((Point, Point))               # Point- <- Point-
# A joint's rate of turning about its axis in space.
Twist = ga.gatype.bivector()                       # Bivector+
# A point quadric maps each point to its polar plane; a dual quadric maps each plane to its pole.
Quadric = ga.gatype((Plane, Point))                # Plane+ <- Point-
DualQuadric = ga.gatype((Point, Plane))            # Point- <- Plane+
# The plane at infinity and the three coordinate planes through the origin.
w = mv.w                                           # Plane+
coordinate_planes = stack([mv.x, mv.y, mv.z])      # [3] Plane+

## Joint twists in the world

The five-part articulated arm has a turret that yaws in the horizontal plane and a shoulder, elbow, and wrist that pitch in the vertical plane. Sandwiching each local rotation plane by its cumulative joint motor places that joint's twist line directly in world coordinates.

In [ ]:
reaching_angles = scenarios.reaching_angles
straight_angles = scenarios.straight_angles

In [ ]:
def arm_axes(joint_angles: tuple[float, float, float, float]) -> tuple[PointMap, Point, Twist]:
    """Body placement maps (`[5] Point- <- Point-`), gripper tip (`[] Point-`), and world joint twists (`[4] Twist+`)."""
    bodies, pivots = arm.robot_arm(joint_angles)
    wrist_pivot = pivots[-1]                                    # [] Motor+
    tip = wrist_pivot >> mv.point([0.0, 0.0, 0.3])              # [] Point-
    # The base turret yaws in xy while the shoulder, elbow, and wrist pitch in zx.
    joint_generators = stack([mv.xy, mv.zx, mv.zx, mv.zx])      # [4] Twist+
    axes = pivots >> joint_generators                           # [4] Twist+
    return bodies, tip, axes


reaching_bodies, reaching_tip, reaching_axes = arm_axes(reaching_angles)

In [ ]:
for index, joint_name in enumerate(("base", "shoulder", "elbow", "wrist")):
    print(f"{joint_name} twist:\n{reaching_axes[index].coefficient_table()}")

## Dual velocity and point force ellipsoids

Each joint twist's commutator with the gripper tip gives an ideal velocity direction, and summing those directions' dyads about the tip forms the velocity ellipsoid as a dual quadric that maps every tangent plane to its contact point. Joining each twist with the tip forms the plane of tip forces that pass through or parallel to that joint axis without loading it; summing those planes' dyads forms the force ellipsoid as a point quadric. Conjugating the dual velocity quadric through the unit sphere centred on the tip turns it directly into the force quadric.

In [ ]:
def velocity_ellipsoid(tip: Point, axes: Twist) -> DualQuadric:
    """Dual quadric (`[] Point- <- Plane+`) of tip velocities reachable with unit joint rates."""
    # Each joint's commutator with the tip gives the ideal velocity direction per unit joint rate.
    velocities = axes.commutator(tip)                           # [...] Point-
    return (velocities * (Plane & velocities)).sum() - tip * (Plane & tip)


def force_ellipsoid(tip: Point, axes: Twist) -> Quadric:
    """Point quadric (`[] Plane+ <- Point-`) of tip forces resisted by unit joint torques."""
    # Joining each joint twist with the tip gives the plane of tip forces that produce zero torque on that joint.
    zero_torque_planes = axes & tip                             # [...] Plane+
    return (zero_torque_planes * (zero_torque_planes & Point)).sum() - w * (w & Point)

In [ ]:
def unit_sphere(centre: Point) -> Quadric:
    """Unit sphere (`[] Plane+ <- Point-`) centred on `centre` as the sum of coordinate-plane dyads through it."""
    central_planes = coordinate_planes - w * (coordinate_planes & centre)   # [3] Plane+
    return (central_planes * (central_planes & Point)).sum() - w * (w & Point)


reaching_velocity = velocity_ellipsoid(reaching_tip, reaching_axes)             # [] Point- <- Plane+
reaching_force = force_ellipsoid(reaching_tip, reaching_axes)                   # [] Plane+ <- Point-
reaching_sphere = unit_sphere(reaching_tip)                                     # [] Plane+ <- Point-
# Conjugate the dual velocity quadric through the tip's unit sphere to obtain its reciprocal point quadric.
reaching_polar = reaching_sphere(reaching_velocity(reaching_sphere(Point)))     # [] Plane+ <- Point-

## Reaching and nearly straight poses

Inverting the dual velocity quadric turns it into a point quadric so the GPU ray tracer renders both ellipsoids as translucent surfaces around the gripper.

In [ ]:
velocity_scale = scenarios.velocity_scale
force_scale = scenarios.force_scale
view_degrees = scenarios.view_degrees
compared_angles = (reaching_angles, straight_angles)

In [ ]:
def pose_ellipsoids(
    joint_angles: tuple[float, float, float, float],
    velocity_scale: float,
    force_scale: float,
) -> tuple[PointMap, Point, Quadric, Quadric]:
    """Body maps (`[5] Point- <- Point-`), tip (`[] Point-`), and scaled velocity and force quadrics (`[] Plane+ <- Point-`)."""
    bodies, tip, axes = arm_axes(joint_angles)
    # Invert the dual velocity quadric into a point quadric so both ellipsoids ray-trace as point surfaces.
    velocity = velocity_ellipsoid(tip, axes * velocity_scale).inverse()     # [] Plane+ <- Point-
    force = force_ellipsoid(tip, axes * force_scale)                        # [] Plane+ <- Point-
    return bodies, tip, velocity, force


static_poses = [pose_ellipsoids(angles, velocity_scale, force_scale) for angles in compared_angles]
save_figure(render.ellipsoids_figure(static_poses, view_degrees), "manipulability");

Each row shows one pose with the scaled blue velocity ellipsoid on the left and the scaled orange force ellipsoid on the right, each panel framed to hold the arm and its surface. In the comfortable reaching pose on top, both ellipsoids stay rounded across all directions; in the nearly straight pose below, the blue velocity ellipsoid flattens into a thin disc transverse to the arm while the orange force ellipsoid elongates into a narrow spindle along it.

## Closed loop through joint space

Sweeping the four joints around a closed sinusoidal loop while keeping the elbow bent away from full extension turns and reshapes both ellipsoids continuously.

In [ ]:
n_frames = scenarios.n_frames
n_pixels = scenarios.n_pixels
frame_duration_ms = scenarios.frame_duration_ms
workspace_centre = scenarios.workspace_centre
workspace_extent = scenarios.workspace_extent
loop_centres = scenarios.loop_centres
loop_amplitudes = scenarios.loop_amplitudes
loop_frequencies = scenarios.loop_frequencies
loop_phases = scenarios.loop_phases

In [ ]:
def joint_loop_poses(
    n_frames: int,
    loop_centres: np.ndarray,
    loop_amplitudes: np.ndarray,
    loop_frequencies: np.ndarray,
    loop_phases: np.ndarray,
    velocity_scale: float,
    force_scale: float,
) -> Iterator[tuple[PointMap, Point, Quadric, Quadric]]:
    """Yields the arm's body maps, tip, and scaled velocity and force quadrics across `n_frames` poses of a closed joint loop."""
    for phase in np.linspace(0.0, 2 * np.pi, n_frames, endpoint=False):
        angles = loop_centres + loop_amplitudes * np.sin(loop_frequencies * phase + loop_phases)
        yield pose_ellipsoids(tuple(angles), velocity_scale, force_scale)


loop_states = show_states(
    joint_loop_poses(n_frames, loop_centres, loop_amplitudes, loop_frequencies, loop_phases, velocity_scale, force_scale),
    render.animate_poses,
    view_degrees,
    workspace_centre,
    workspace_extent,
    n_pixels,
    name="manipulability_sweep",
    duration_ms=frame_duration_ms,
);

As the turret yaws and the elbow bends and extends around the workspace, every direction along which the blue velocity ellipsoid contracts sees the orange force ellipsoid stretch in inverse proportion.

In [ ]:
# checks
basis_points = stack([mv.w, mv.x, mv.y, mv.z]).dual()                           # [4] Point-
# In the reaching pose, the force ellipsoid equals the polar of the velocity ellipsoid in the tip's unit sphere.
reaching_residual = (reaching_force - reaching_polar)(basis_points[None, :]) & basis_points[:, None]  # [4, 4] Scalar+
assert (reaching_residual.abs() < 1e-8).all()

# Across both static poses and the streamed loop poses, removing the display scales recovers exact polar reciprocity.
for _, pose_tip, scaled_velocity, scaled_force in [*static_poses, loop_states[0], loop_states[n_frames // 2]]:
    pose_sphere = unit_sphere(pose_tip)                                         # [] Plane+ <- Point-
    unscaled_velocity = (
        (scaled_velocity.inverse() + pose_tip * (Plane & pose_tip)) / (velocity_scale**2)
        - pose_tip * (Plane & pose_tip)
    )                                                                           # [] Point- <- Plane+
    unscaled_force = (
        (scaled_force + w * (w & Point)) / (force_scale**2) - w * (w & Point)
    )                                                                           # [] Plane+ <- Point-
    polar_residual = (unscaled_force - pose_sphere(unscaled_velocity(pose_sphere(Point))))(basis_points[None, :]) & basis_points[:, None]  # [4, 4] Scalar+
    assert (polar_residual.abs() < 1e-8).all()

# Unit tip forces evaluate on the force quadric to squared joint torque minus one.
unit_directions = scenarios.sample_directions / scenarios.sample_directions.antinorm()  # [2] Point-
reaching_torques = reaching_axes[:, None] & (reaching_tip & unit_directions)    # [4, 2] Scalar+
torque_norm_sq = reaching_torques.squared().sum(axis=0)                         # [2] Scalar+
loaded_points = reaching_tip + unit_directions                                  # [2] Point-
assert (((reaching_force(loaded_points) & loaded_points) - (torque_norm_sq - 1)).abs() < 1e-8).all()